In [ ]:
import dotenv
dotenv.load_dotenv()

In [ ]:
import os
from logfire.query_client import LogfireQueryClient

read_token = os.getenv('LOGFIRE_READ_TOKEN')
logfire_query_client = LogfireQueryClient(read_token=read_token)

In [ ]:
trace_rows = logfire_query_client.query_json_rows(
    sql="""
    SELECT
        trace_id,
        start_timestamp,
        duration
    FROM records
    WHERE span_name = 'streamlit_session'
    ORDER BY start_timestamp DESC
    LIMIT 5
    """
)

In [ ]:
trace_ids = [r['trace_id'] for r in trace_rows['rows']]
trace_ids

In [ ]:
trace_id = '019e04e4b3618c7cc2f830142d0b330b'

In [ ]:
run_row = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        attributes->'pydantic_ai.all_messages' as all_messages,
        attributes->>'gen_ai.usage.input_tokens' as input_tokens,
        attributes->>'gen_ai.usage.output_tokens' as output_tokens
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'agent run'
    ORDER BY start_timestamp DESC
    LIMIT 1
    """
)

In [ ]:
all_messages = run_row['rows'][0]

In [ ]:
all_messages['all_messages'][0]

In [ ]:
import sys
sys.path.append('..')

In [ ]:
import trace_replay

In [ ]:
trace = trace_replay.fetch_trace(trace_id, logfire_query_client)

In [ ]:
run = trace_replay.trace_to_run_result(trace)

In [ ]:
run.output

In [ ]:
traces = trace_replay.fetch_traces(trace_ids, logfire_query_client)

In [ ]:
runs = []


for trace in traces.values():
    run = trace_replay.trace_to_run_result(trace)
    runs.append(run)

In [23]:
import os
os.makedirs('data', exist_ok=True)

import pickle
with open('data/logs.bin', 'wb') as f_out:
    pickle.dump(runs, f_out)

In [25]:
with open('data/logs.bin', 'rb') as f_in:
    runs = pickle.load(f_in)

In [26]:
runs

[AgentRunResult(output={'checks': [{'explanation': 'All information regarding changing LLM models for built-in judges, including examples and supported providers, is directly from the provided `metrics/customize_llm_judge.mdx` document.', 'rule': 'The answer must only use facts found in the documentation knowledge base.', 'passed': True}, {'passed': True, 'explanation': "The answer explicitly states 'Yes, you can use a different LLM model for built-in judges' and then provides clear instructions on how to do so.", 'rule': "The answer must directly address the user's question."}, {'explanation': 'No outside information or assumptions were introduced; the answer is strictly based on the content of the provided documentation.', 'rule': 'The answer must not introduce outside knowledge or assumptions.', 'passed': True}, {'explanation': 'The answer was found and directly addresses the question.', 'rule': 'If the answer cannot be found in the retrieved documents, clearly inform the user.', 'p